In [1]:
import pickle, lzma
import numpy as np
import scipy.integrate as spint
from jLM.Solvers import ConstBoundaryConc, makeSolver
from lm import MGPUMpdRdmeSolver, MpdRdmeSolver,IntMpdRdmeSolver
from jLM.RDME import Sim as RDMESim
from jLM.RDME import File as RDMEFile
from jLM.RegionBuilder import RegionBuilder
import jLM

# Initialization and Spatial Geometry

In [2]:
latticeData = pickle.load(lzma.open("yeast-lattice.2.pkl.xz", "rb"))

siteMap = {n:i for i,n in enumerate(latticeData['names'])}
def boolLattice(x):
    return latticeData['lattice'] == siteMap[x]

extracellular = boolLattice("extracellular") | boolLattice("cellWall")
obstructions = boolLattice("nuclearEnvelope") | boolLattice("mitochondria") | boolLattice('vacuole')
ribosomes = boolLattice("ribosomes")
membrane = boolLattice("plasmaMembrane")
nucleus = boolLattice("nucleoplasm") | boolLattice("nuclearPores")
cytoplasm = boolLattice("cytoplasm")

decimation = latticeData['decimation']

In [3]:
sim = RDMESim("Galactose switch, RDME/ODE hybrid",
              "icLattice.lm",
              latticeData['lattice'].shape,
              latticeData['latticeSpacing'],
              "extracellular")

In [4]:
B = RegionBuilder(sim)

B.compose((sim.region('extracellular'), extracellular),
          (sim.region('obstructions'), obstructions),
          (sim.region('plasmaMembrane'), membrane),
          (sim.region('cytoplasm'), cytoplasm),
          (sim.region('nucleoplasm'), nucleus),
          (sim.region('ribosomes'), ribosomes))

abbreviation for object access

In [5]:
sp = sim.sp   # species object access
reg = sim.reg # region object access
rc = sim.rc   # rate constant object access
dc = sim.dc   # diffusion constant object access

# Simulation parameters

In [6]:
sim.simulationTime = 5 # seconds
sim.timestep = 67e-6 # seconds
# mimic the old way to define the interval
interval = 0.05 # seconds
sim.latticeWriteInterval= int(interval/sim.timestep) # .05s
sim.speciesWriteInterval= int(interval/sim.timestep) # .05s
sim.hookInterval= int(interval/sim.timestep) #.05s

# initial conditions for external galactose 
externalGal = 11.1e-3 # M   
# Number of ribosomes in total
nRibosomes = np.sum(sim.siteLattice == reg.ribosomes.idx)
# diffusion coefficients for mRNA
# mRNADiffusion = 0.5e-12 # m^2/s
mRNADiffusion = 0.05e-12 # 2018, IET

# Species Definitions

## Reporter GFP

In [7]:
with sim.construct():
    sim.species('DGrep', texRepr='D_{rep}', annotation="Reporter gene (inactive)")
    sim.species('DGrep_G4d', texRepr='D_{rep}{:}G_{4D}', annotation="Reporter gene activated")
    sim.species('DGrep_G4d_G80d', texRepr='D_{rep}{:}G_{4D}{:}G_{80D}', annotation="Reporter gene repressed")
    sim.species('Rrep', texRepr='R_{rep}', annotation="Reporter mRNA")
    sim.species('Grep', texRepr='G_{rep}', annotation="Reporter GFP")
    
    

Index,Name,Annotation
1,DGrep,Reporter gene (inactive)
2,DGrep_G4d,Reporter gene activated
3,DGrep_G4d_G80d,Reporter gene repressed
4,Rrep,Reporter mRNA
5,Grep,Reporter GFP


## GAL1 (G1) metabolism/ degradation process

In [8]:
with sim.construct():
    sim.species('DG1', texRepr='D_{G1}', annotation="Galactose metabolism gene (inactive)")
    sim.species('DG1_G4d', texRepr='D_{G1}{:}G_{4D}', annotation="Galactose metabolism gene activated")
    sim.species('DG1_G4d_G80d', texRepr='D_{G1}{:}G_{4D}{:}G_{80D}', annotation="Galactose metabolism gene repressed")
    sim.species('R1', texRepr='R_{1}', annotation="Galactose metabolism mRNA")
    sim.species('G1', texRepr='G_{1}', annotation="Galactose metabolism protein")

Index,Name,Annotation
6,DG1,Galactose metabolism gene (inactive)
7,DG1_G4d,Galactose metabolism gene activated
8,DG1_G4d_G80d,Galactose metabolism gene repressed
9,R1,Galactose metabolism mRNA
10,G1,Galactose metabolism protein


## GAL2(G2) transporter

In [9]:
with sim.construct():
    sim.species('DG2', texRepr='D_{G2}', annotation="Galactose transport gene (inactive)")
    sim.species('DG2_G4d', texRepr='D_{G2}{:}G_{4D}', annotation="Galactose transport gene activated")
    sim.species('DG2_G4d_G80d', texRepr='D_{G2}{:}G_{4D}{:}G_{80D}', annotation="Galactose transport gene repressed")
    sim.species('R2', texRepr='R_{2}', annotation="Galactose transport mRNA")
    sim.species('G2', texRepr='G_{2}', annotation="Galactose transport protein")

Index,Name,Annotation
11,DG2,Galactose transport gene (inactive)
12,DG2_G4d,Galactose transport gene activated
13,DG2_G4d_G80d,Galactose transport gene repressed
14,R2,Galactose transport mRNA
15,G2,Galactose transport protein


## Gal3(regulatory protein)

In [10]:
with sim.construct():
    sim.species('DG3', texRepr='D_{G3}', annotation="Gal3 gene (inactive)")
    sim.species('DG3_G4d', texRepr='D_{G3}{:}G_{4D}', annotation="Gal3 gene activated")
    sim.species('DG3_G4d_G80d', texRepr='D_{G3}{:}G_{4D}{:}G_{80D}', annotation="Gal3 gene repressed")
    sim.species('R3', texRepr='R_{3}', annotation="Gal3 mRNA")
    sim.species('G3', texRepr='G_{3}', annotation="Gal3 protein")
    sim.species('G3i', texRepr='G_{3i}', annotation="activated Gal3 bound to galactose ")

Index,Name,Annotation
16,DG3,Gal3 gene (inactive)
17,DG3_G4d,Gal3 gene activated
18,DG3_G4d_G80d,Gal3 gene repressed
19,R3,Gal3 mRNA
20,G3,Gal3 protein
21,G3i,activated Gal3 bound to galactose


## GAL4(G4)

In [11]:
with sim.construct():
    sim.species('DG4', texRepr='D_{G4}', annotation="Gal4 gene (inactive)")
    sim.species('R4', texRepr='R_{4}', annotation="Gal4 mRNA")
    sim.species('G4', texRepr='G_{4}', annotation="Gal4 protein")
    sim.species('G4d', texRepr='G_{4D}', annotation="Gal4 dimer")

Index,Name,Annotation
22,DG4,Gal4 gene (inactive)
23,R4,Gal4 mRNA
24,G4,Gal4 protein
25,G4d,Gal4 dimer


## GAL80 (G80)

In [12]:
with sim.construct():
    sim.species('DG80', texRepr='D_{G80}', annotation="Gal80 gene (inactive)")
    sim.species('DG80_G4d', texRepr='D_{G80}{:}G_{4D}', annotation="Gal80 gene activated")
    sim.species('DG80_G4d_G80d', texRepr='D_{G80}{:}G_{4D}{:}G_{80D}', annotation="Gal80 gene repressed")
    sim.species('R80', texRepr='R_{80}', annotation="Gal80 mRNA")
    sim.species('G80', texRepr='G_{80}', annotation="Gal80 protein")
    sim.species('G80d', texRepr='G_{80D}', annotation="Gal80 dimer")
    sim.species('G80d_G3i', texRepr='G_{80D}{:}G_{3i}', annotation="Gal80 dimer bound to activated Gal3")

Index,Name,Annotation
26,DG80,Gal80 gene (inactive)
27,DG80_G4d,Gal80 gene activated
28,DG80_G4d_G80d,Gal80 gene repressed
29,R80,Gal80 mRNA
30,G80,Gal80 protein
31,G80d,Gal80 dimer
32,G80d_G3i,Gal80 dimer bound to activated Gal3


## Ribosomes Particles

In [13]:
with sim.construct():
    sim.species('ribosome', texRepr='Ribosome', annotation="Ribosome (inactive)")
    sim.species('ribosomeR1', texRepr='Ribosome{:}R_{1}', annotation="Ribosome bound to Gal1 mRNA")
    sim.species('ribosomeR2', texRepr='Ribosome{:}R_{2}', annotation="Ribosome bound to Gal2 mRNA")
    sim.species('ribosomeR3', texRepr='Ribosome{:}R_{3}', annotation="Ribosome bound to Gal3 mRNA")
    sim.species('ribosomeR4', texRepr='Ribosome{:}R_{4}', annotation="Ribosome bound to Gal4 mRNA")
    sim.species('ribosomeR80', texRepr='Ribosome{:}R_{80}', annotation="Ribosome bound to Gal80 mRNA")
    sim.species('ribosomeGrep', texRepr='Ribosome{:}G_{rep}', annotation="Ribosome bound to reporter mRNA")

Index,Name,Annotation
33,ribosome,Ribosome (inactive)
34,ribosomeR1,Ribosome bound to Gal1 mRNA
35,ribosomeR2,Ribosome bound to Gal2 mRNA
36,ribosomeR3,Ribosome bound to Gal3 mRNA
37,ribosomeR4,Ribosome bound to Gal4 mRNA
38,ribosomeR80,Ribosome bound to Gal80 mRNA
39,ribosomeGrep,Ribosome bound to reporter mRNA


# 3. Reactions

In [14]:
cellVol = 3.57e-14 # L, cell size from Ramsey paper SI, haploid yeast
nav = cellVol*6.022e23  # volume times Avogadro's number
invMin2invSec = 1/60.0 # s^-1/min^-1
conv2ndOrder = invMin2invSec*nav # convert to s^-1*M^-1
conv1stOrder = invMin2invSec # convert to s^-1

## Dimerization

In [15]:
with sim.construct():
    sim.rateConst('fd',100 * conv2ndOrder, order= 2, annotation ="Gal4p/Gal80p dimer formation")
    sim.rateConst('rd', 0.001 * conv1stOrder, order= 1, annotation ="Gal4p/Gal80p dimer dissociation")
    sim.reaction([sp.G4, sp.G4], [sp.G4d], rc.fd, annotation ="Gal4p/Gal80p dimer formation", regions=[reg.cytoplasm, reg.nucleoplasm])
    sim.reaction([sp.G4d], [sp.G4, sp.G80], rc.rd, annotation ="Gal4p/Gal80p dimer dissociation", regions=[reg.cytoplasm, reg.nucleoplasm])
    sim.reaction([sp.G80, sp.G4d], [sp.G80d], rc.fd, annotation ="Gal80p/Gal80p dimer formation", regions=[reg.cytoplasm, reg.nucleoplasm])
    sim.reaction([sp.G80d], [sp.G80, sp.G80], rc.rd, annotation ="Gal80p/Gal80p dimer dissociation", regions=[reg.cytoplasm, reg.nucleoplasm])

## DNA regulation

In [16]:
with sim.construct():
    Kp4 = 0.2600 # 4 binding sites
    Kq4 = 1.1721 
    
    kf1_4 = 0.1
    kf2_4 = 0.1
    kr1_4 = kf1_4/Kp4
    kr2_4 = kf2_4/Kq4
    
    Kp5 = 0.0099 # 5 binding sites
    Kq5 = 0.7408 
    
    kf1_5 = 0.1
    kf2_5 = 0.1
    kr1_5 = kf1_5/Kp5
    kr2_5 = kf2_5/Kq5
    
    
    Kp = 0.0248 # 1 binding site
    Kq = 0.1885 
    kf1 = 0.1
    kr1 = kf1/Kp
    kf2 = 0.1
    kr2 = kf2/Kq

    #convBinding = invMin2invSec*sim.siteNAV

    sim.rateConst("f1", kf1*conv2ndOrder, order=2, annotation="Gene/Gal4p binding [1 site]")
    sim.rateConst("r1", kr1*conv1stOrder, order=1, annotation="Gene/Gal4p dissociation [1 site]")
    sim.rateConst("f2", kf2*conv2ndOrder, order=2, annotation="Gene/Gal80p binding [1 site]")
    sim.rateConst("r2", kr2*conv1stOrder, order=1, annotation="Gene/Gal80p dissociation [1 site]")
    
    sim.rateConst("f1_4", kf1_4*conv2ndOrder, order=2, annotation="Gene/Gal4p binding [4 sites]")
    sim.rateConst("r1_4", kr1_4*conv1stOrder, order=1, annotation="Gene/Gal4p dissociation [4 sites]")
    sim.rateConst("f2_4", kf2_4*conv2ndOrder, order=2, annotation="Gene/Gal80p binding [4 sites]")
    sim.rateConst("r2_4", kr2_4*conv1stOrder, order=1, annotation="Gene/Gal80p dissociation [4 sites]")
    
    sim.rateConst("f1_5", kf1_5*conv2ndOrder, order=2, annotation="Gene/Gal4p binding [5 sites]")
    sim.rateConst("r1_5", kr1_5*conv1stOrder, order=1, annotation="Gene/Gal4p dissociation [5 sites]")
    sim.rateConst("f2_5", kf2_5*conv2ndOrder, order=2, annotation="Gene/Gal80p binding [5 sites]")
    sim.rateConst("r2_5", kr2_5*conv1stOrder, order=1, annotation="Gene/Gal80p dissociation [5 sites]")
    
    ### details
    # G1, Grep has 4 sites
    # G2 has 5 sites
    # G3, G80 have 1 site
    
    dnas            = [sp.DG1           , sp.DG2        , sp.DG3            ,  sp.DG80          , sp.DGrep]
    # mrnas           = [sp.R1            , sp.R2         , sp.R3,  sp.R80, sp.Rrep]
    dna_gal4        = [sp.DG1_G4d       , sp.DG2_G4d    , sp.DG3_G4d        ,  sp.DG80_G4d      , sp.DGrep_G4d]
    dna_gal4_gal80  = [sp.DG1_G4d_G80d, sp.DG2_G4d_G80d , sp.DG3_G4d_G80d   , sp.DG80_G4d_G80d  , sp.DGrep_G4d_G80d]
    f_g4s           = [rc.f1_4          , rc.f1_5       , rc.f1             , rc.f1             , rc.f1_4]
    r_g4s           = [rc.r1_4          , rc.r1_5       , rc.r1             , rc.r1             , rc.r1_4]
    f_g4g80s        = [rc.f2_4          , rc.f2_5       , rc.f2             , rc.f2             , rc.f2_4]
    r_g4g80s        = [rc.r2_4          , rc.r2_5       , rc.r2             , rc.r2             , rc.r2_4]
    
    for dna, dna_gal4, dna_gal4_gal80, f_g4, r_g4, f_g4g80, r_g4g80 in zip(dnas, dna_gal4, dna_gal4_gal80, f_g4s, r_g4s, f_g4g80s, r_g4g80s):
        sim.reaction([dna, sp.G4], [dna_gal4], f_g4, annotation="Gal4p binding to gene", regions=reg.nucleoplasm)
        sim.reaction([dna_gal4], [dna, sp.G4], r_g4, annotation="Gal4p dissociation from gene", regions=reg.nucleoplasm)
        sim.reaction([dna_gal4, sp.G80], [dna_gal4_gal80], f_g4g80, annotation="Gal80p binding to gene", regions=reg.nucleoplasm)
        sim.reaction([dna_gal4_gal80], [dna_gal4, sp.G80], r_g4g80, annotation="Gal80p dissociation from gene", regions=reg.nucleoplasm)
    

Index,Reaction,Rate,Regions,Annotation
4,$$D_{G1} + G_{4}\overset{k_{\mathrm{f1\_4}}}{\longrightarrow}D_{G1}{:}G_{4D}$$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal4p binding to gene
5,$$D_{G1}{:}G_{4D}\overset{k_{\mathrm{r1\_4}}}{\longrightarrow}D_{G1} + G_{4}$$,6.410 × 10-3 s-1,nucleoplasm,Gal4p dissociation from gene
6,$$D_{G1}{:}G_{4D} + G_{80}\overset{k_{\mathrm{f2\_4}}}{\longrightarrow}D_{G1}{:}G_{4D}{:}G_{80D}$$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal80p binding to gene
7,$$D_{G1}{:}G_{4D}{:}G_{80D}\overset{k_{\mathrm{r2\_4}}}{\longrightarrow}D_{G1}{:}G_{4D} + G_{80}$$,1.422 × 10-3 s-1,nucleoplasm,Gal80p dissociation from gene
8,$$D_{G2} + G_{4}\overset{k_{\mathrm{f1\_5}}}{\longrightarrow}D_{G2}{:}G_{4D}$$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal4p binding to gene
9,$$D_{G2}{:}G_{4D}\overset{k_{\mathrm{r1\_5}}}{\longrightarrow}D_{G2} + G_{4}$$,0.168 s-1,nucleoplasm,Gal4p dissociation from gene
10,$$D_{G2}{:}G_{4D} + G_{80}\overset{k_{\mathrm{f2\_5}}}{\longrightarrow}D_{G2}{:}G_{4D}{:}G_{80D}$$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal80p binding to gene
11,$$D_{G2}{:}G_{4D}{:}G_{80D}\overset{k_{\mathrm{r2\_5}}}{\longrightarrow}D_{G2}{:}G_{4D} + G_{80}$$,2.250 × 10-3 s-1,nucleoplasm,Gal80p dissociation from gene
12,$$D_{G3} + G_{4}\overset{k_{\mathrm{f1}}}{\longrightarrow}D_{G3}{:}G_{4D}$$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal4p binding to gene
13,$$D_{G3}{:}G_{4D}\overset{k_{\mathrm{r1}}}{\longrightarrow}D_{G3} + G_{4}$$,0.067 s-1,nucleoplasm,Gal4p dissociation from gene


## G3 activation

In [17]:
with sim.construct():
    sim.rateConst("fi", 7.45e-7*conv2ndOrder, order=1, annotation="Gal3p activation ") # needs to be multiplied by [GAI] in mol/L,  modified by set reactions rate in hook 
    sim.rateConst("ri", 890.0*conv1stOrder, order=1, annotation="Gal3p deactivation")
    sim.rateConst("fd3i80", 0.025716*conv2ndOrder, order=2, annotation="Gal3p*/Gal80 association")
    sim.rateConst("dr3i80", 0.0159616*conv1stOrder, order=1, annotation="Gal3p*/Gal80 disassociation")
    sim.rateConst("dp_gal3", 0.01155*conv1stOrder, order=1, annotation="GAL3 degradation")
    sim.rateConst("dp_gal3gal80", 0.5*rc.dp_gal3.value, order=1, annotation="Gal3p*:Gal80 degradation")
    
    sim.reaction(sp.G3, sp.G3i, rc.fi, annotation="Gal3p activation", regions=reg.cytoplasm)
    sim.reaction(sp.G3i, sp.G3, rc.ri, annotation="Gal3p deactivation", regions=reg.cytoplasm)
    ## Gal3p* Gal80p
    sim.reaction([sp.G3i, sp.G80], [sp.G80d_G3i], rc.fd3i80, annotation="Gal3p*/Gal80 association", regions=reg.cytoplasm)
    sim.reaction([sp.G80d_G3i], [sp.G3i, sp.G80], rc.dr3i80, annotation="Gal3p*/Gal80 disassociation", regions=reg.cytoplasm)
    ## Gal3p+galactose degradation, should left the internal galactose, but neglected
    sim.reaction(sp.G3i, [], rc.dp_gal3, annotation="GAL3 degradation", regions=reg.cytoplasm)
    sim.reaction(sp.G80d_G3i, [], rc.dp_gal3gal80, annotation="Gal3p*:Gal80 degradation", regions=reg.cytoplasm)

## Transcription

In [18]:
with sim.construct():
    sim.rateConst("alpha1", 0.7379*conv1stOrder, order=1, annotation='GAL1 transcription')
    sim.rateConst("alpha2", 2.542*conv1stOrder, order=1, annotation='GAL2 transcription')
    sim.rateConst("alpha3", 0.571429*0.7465*conv1stOrder, order=1, annotation='GAL3 transcription')
    sim.rateConst("ir_gal4", 0.009902*conv1stOrder, order=1, annotation='GAL4 transcription')
    sim.rateConst("alpha_rep", 1.1440*conv1stOrder, order=1, annotation='GFP transcription')
    sim.rateConst("alpha80", 0.6065*conv1stOrder, order=1, annotation='GAL80 transcription')

    sim.rateConst("dr_gal1", 0.02236*conv1stOrder, order=1, annotation='GAL1 mRNA degradation')
    sim.rateConst("dr_gal2", 0.07702*conv1stOrder, order=1, annotation='GAL2 mRNA degradation')
    sim.rateConst("dr_gal3", 0.02666*conv1stOrder, order=1, annotation='GAL3 mRNA degradation')
    sim.rateConst("dr_gal4", 0.02476*conv1stOrder, order=1, annotation='GAL4 mRNA degradation')
    sim.rateConst("dr_rep", 0.03466*conv1stOrder, order=1, annotation='GFP mRNA degradation')
    sim.rateConst("dr_gal80", 0.02888*conv1stOrder, order=1, annotation='GAL80 mRNA degradation')
    
    transcription_rates = [rc.alpha1,   rc.alpha2,  rc.alpha3,  rc.ir_gal4, rc.alpha_rep,   rc.alpha80]
    decay_rates         = [rc.dr_gal1,  rc.dr_gal2, rc.dr_gal3, rc.dr_gal4, rc.dr_rep,      rc.dr_gal80]
    genes               = [sp.DG1_G4d,  sp.DG2_G4d, sp.DG3_G4d, sp.DG4,     sp.DGrep_G4d,   sp.DG80_G4d]
    mrnas               = [sp.R1,       sp.R2,      sp.R3,      sp.R4,      sp.Rrep,        sp.R80]
    
    for trans_rate, decay_rate, gene, mrna in zip(transcription_rates, decay_rates, genes, mrnas):
        sim.reaction([gene], [gene, mrna], trans_rate, regions=reg.nucleoplasm)
        sim.reaction([mrna], [], decay_rate, regions=[reg.nucleoplasm, reg.cytoplasm, reg.ribosomes])

Index,Reaction,Rate,Regions,Annotation
30,$$D_{G1}{:}G_{4D}\overset{k_{\mathrm{alpha1}}}{\longrightarrow}D_{G1}{:}G_{4D} + R_{1}$$,0.012 s-1,nucleoplasm,GAL1 transcription
31,$$R_{1}\overset{k_{\mathrm{dr\_gal1}}}{\longrightarrow}\varnothing$$,3.727 × 10-4 s-1,"nucleoplasm, cytoplasm, ribosomes",GAL1 mRNA degradation
32,$$D_{G2}{:}G_{4D}\overset{k_{\mathrm{alpha2}}}{\longrightarrow}D_{G2}{:}G_{4D} + R_{2}$$,0.042 s-1,nucleoplasm,GAL2 transcription
33,$$R_{2}\overset{k_{\mathrm{dr\_gal2}}}{\longrightarrow}\varnothing$$,1.284 × 10-3 s-1,"nucleoplasm, cytoplasm, ribosomes",GAL2 mRNA degradation
34,$$D_{G3}{:}G_{4D}\overset{k_{\mathrm{alpha3}}}{\longrightarrow}D_{G3}{:}G_{4D} + R_{3}$$,7.110 × 10-3 s-1,nucleoplasm,GAL3 transcription
35,$$R_{3}\overset{k_{\mathrm{dr\_gal3}}}{\longrightarrow}\varnothing$$,4.443 × 10-4 s-1,"nucleoplasm, cytoplasm, ribosomes",GAL3 mRNA degradation
36,$$D_{G4}\overset{k_{\mathrm{ir\_gal4}}}{\longrightarrow}D_{G4} + R_{4}$$,1.650 × 10-4 s-1,nucleoplasm,GAL4 transcription
37,$$R_{4}\overset{k_{\mathrm{dr\_gal4}}}{\longrightarrow}\varnothing$$,4.127 × 10-4 s-1,"nucleoplasm, cytoplasm, ribosomes",GAL4 mRNA degradation
38,$$D_{rep}{:}G_{4D}\overset{k_{\mathrm{alpha\_rep}}}{\longrightarrow}D_{rep}{:}G_{4D} + R_{rep}$$,0.019 s-1,nucleoplasm,GFP transcription
39,$$R_{rep}\overset{k_{\mathrm{dr\_rep}}}{\longrightarrow}\varnothing$$,5.777 × 10-4 s-1,"nucleoplasm, cytoplasm, ribosomes",GFP mRNA degradation


transcription -2, dummy related transcription and degradation.

## Translation

Ribosome/mRNA association rate. For the diffusive propensity to be equal to the reaction propensity, we need
$$ k = 2000DN_A\lambda.$$
We will choose the reaction propensity to be 0.2 of the diffsive propensity so that immediate reassociation happens 1 of 5 dissociation events.

In [19]:
with sim.construct():
    #tlInitDet = 100e6 # 10.1016/j.molcel.2006.02.014 [eco]
    tlInitDet = 0.2 * 2000 * mRNADiffusion * sim.NA * sim.latticeSpacing
    sim.rateConst("rib_assoc", tlInitDet, order=2, annotation='mRNA/Ribosome association rate') 
    
    # translation rates
    sim.rateConst("ip_gal1", 1.9254*conv1stOrder, order=1, annotation='GAL1 translation')
    sim.rateConst("ip_gal2", 13.4779*conv1stOrder, order=1, annotation="GAL2 translation")
    sim.rateConst("ip_gal3", 55.4518*conv1stOrder, order=1, annotation="GAL3 translation")
    sim.rateConst("ip_gal4", 10.7091*conv1stOrder, order=1, annotation="GAL4 translation")
    sim.rateConst("ip_rep", 5.7762*conv1stOrder, order=1, annotation="GFP translation")
    sim.rateConst("ip_gal80", 3.6737*conv1stOrder, order=1, annotation="GAL80 translation")
    # protein degradation rates
    sim.rateConst("dp_gal1", 0.003851*conv1stOrder, order=1, annotation='GAL1 degradation')
    sim.rateConst("dp_gal2", 0.003851*conv1stOrder, order=1, annotation="GAL2 degradation")
    sim.rateConst("dp_gal3", 0.01155*conv1stOrder, order=1, annotation="GAL3 degradation")
    sim.rateConst("dp_gal4", 0.006931*conv1stOrder, order=1, annotation="GAL4 degradation")
    sim.rateConst("dp_rep", 0.01155*conv1stOrder, order=1, annotation="GFP degradation")
    sim.rateConst("dp_gal80", 0.006931*conv1stOrder, order=1, annotation="GAL80 degradation")
    #rates and species
    ktls =                 [rc.ip_gal1,     rc.ip_gal2,     rc.ip_gal3,     rc.ip_gal4,     rc.ip_rep,      rc.ip_gal80]
    dcys =                 [rc.dp_gal1,     rc.dp_gal2,     rc.dp_gal3,     rc.dp_gal4,     rc.dp_rep,      rc.dp_gal80] 
    mdcys =                [rc.dr_gal1,     rc.dr_gal2,     rc.dr_gal3,     rc.dr_gal4,     rc.dr_rep,      rc.dr_gal80]
    translatingRibosomes = [sp.ribosomeR1, sp.ribosomeR2, sp.ribosomeR3, sp.ribosomeR4, sp.ribosomeGrep, sp.ribosomeR80]
    prots =                [sp.G1,         sp.G2,         sp.G3,         sp.G4,         sp.Grep,         sp.G80]
    
    for mrna, translatingRibosomes, protein, ktl, dcy, mdcy in zip(mrnas, translatingRibosomes, prots, ktls, dcys, mdcys):
        #association
        sim.reaction([sp.ribosome, mrna], [translatingRibosomes], rc.rib_assoc, regions=reg.ribosomes)
        #translation
        sim.reaction([translatingRibosomes], [sp.ribosome, mrna, protein], ktl, regions=reg.ribosomes)
        #degradation in association form
        sim.reaction([translatingRibosomes], [sp.ribosome], mdcy, regions=reg.ribosomes)
    
    

Index,Reaction,Rate,Regions,Annotation
42,$$Ribosome + R_{1}\overset{k_{\mathrm{rib\_assoc}}}{\longrightarrow}Ribosome{:}R_{1}$$,3.454 × 105 M-1⋅s-1,ribosomes,mRNA/Ribosome association rate
43,$$Ribosome{:}R_{1}\overset{k_{\mathrm{ip\_gal1}}}{\longrightarrow}Ribosome + R_{1} + G_{1}$$,0.032 s-1,ribosomes,GAL1 translation
44,$$Ribosome{:}R_{1}\overset{k_{\mathrm{dr\_gal1}}}{\longrightarrow}Ribosome$$,3.727 × 10-4 s-1,ribosomes,GAL1 mRNA degradation
45,$$Ribosome + R_{2}\overset{k_{\mathrm{rib\_assoc}}}{\longrightarrow}Ribosome{:}R_{2}$$,3.454 × 105 M-1⋅s-1,ribosomes,mRNA/Ribosome association rate
46,$$Ribosome{:}R_{2}\overset{k_{\mathrm{ip\_gal2}}}{\longrightarrow}Ribosome + R_{2} + G_{2}$$,0.225 s-1,ribosomes,GAL2 translation
47,$$Ribosome{:}R_{2}\overset{k_{\mathrm{dr\_gal2}}}{\longrightarrow}Ribosome$$,1.284 × 10-3 s-1,ribosomes,GAL2 mRNA degradation
48,$$Ribosome + R_{3}\overset{k_{\mathrm{rib\_assoc}}}{\longrightarrow}Ribosome{:}R_{3}$$,3.454 × 105 M-1⋅s-1,ribosomes,mRNA/Ribosome association rate
49,$$Ribosome{:}R_{3}\overset{k_{\mathrm{ip\_gal3}}}{\longrightarrow}Ribosome + R_{3} + G_{3}$$,0.924 s-1,ribosomes,GAL3 translation
50,$$Ribosome{:}R_{3}\overset{k_{\mathrm{dr\_gal3}}}{\longrightarrow}Ribosome$$,4.443 × 10-4 s-1,ribosomes,GAL3 mRNA degradation
51,$$Ribosome + R_{4}\overset{k_{\mathrm{rib\_assoc}}}{\longrightarrow}Ribosome{:}R_{4}$$,3.454 × 105 M-1⋅s-1,ribosomes,mRNA/Ribosome association rate


-3, dummy rna association, translation and degradation

## Protein degradation(out of ribosome)

In [20]:
with sim.construct():
    deg_compartments =         [[reg.ribosomes, reg.cytoplasm], 
                                [reg.ribosomes, reg.cytoplasm, reg.plasmaMembrane], 
                                [reg.ribosomes, reg.cytoplasm], 
                                [reg.ribosomes, reg.cytoplasm, reg.nucleoplasm], 
                                [reg.ribosomes, reg.cytoplasm], 
                                [reg.ribosomes, reg.cytoplasm, reg.nucleoplasm]]
    for protein, decay_rate, region in zip(prots, dcys, deg_compartments):
        sim.reaction([protein], [], decay_rate, regions=region)


Index,Reaction,Rate,Regions,Annotation
60,$$G_{1}\overset{k_{\mathrm{dp\_gal1}}}{\longrightarrow}\varnothing$$,6.418 × 10-5 s-1,"ribosomes, cytoplasm",GAL1 degradation
61,$$G_{2}\overset{k_{\mathrm{dp\_gal2}}}{\longrightarrow}\varnothing$$,6.418 × 10-5 s-1,"ribosomes, cytoplasm, plasmaMembrane",GAL2 degradation
62,$$G_{3}\overset{k_{\mathrm{dp\_gal3}}}{\longrightarrow}\varnothing$$,1.925 × 10-4 s-1,"ribosomes, cytoplasm",GAL3 degradation
63,$$G_{4}\overset{k_{\mathrm{dp\_gal4}}}{\longrightarrow}\varnothing$$,1.155 × 10-4 s-1,"ribosomes, cytoplasm, nucleoplasm",GAL4 degradation
64,$$G_{rep}\overset{k_{\mathrm{dp\_rep}}}{\longrightarrow}\varnothing$$,1.925 × 10-4 s-1,"ribosomes, cytoplasm",GFP degradation
65,$$G_{80}\overset{k_{\mathrm{dp\_gal80}}}{\longrightarrow}\varnothing$$,1.155 × 10-4 s-1,"ribosomes, cytoplasm, nucleoplasm",GAL80 degradation


In [21]:
sim.showReactions()

Index,Reaction,Rate,Regions,Description
0,$G_{4} + G_{4}\overset{k_{\mathrm{fd}}}{\longrightarrow}G_{4D}$,3.583 × 1010 M-1⋅s-1,"cytoplasm, nucleoplasm",Gal4p/Gal80p dimer formation
1,$G_{4D}\overset{k_{\mathrm{rd}}}{\longrightarrow}G_{4} + G_{80}$,1.667 × 10-5 s-1,"cytoplasm, nucleoplasm",Gal4p/Gal80p dimer dissociation
2,$G_{80} + G_{4D}\overset{k_{\mathrm{fd}}}{\longrightarrow}G_{80D}$,3.583 × 1010 M-1⋅s-1,"cytoplasm, nucleoplasm",Gal80p/Gal80p dimer formation
3,$G_{80D}\overset{k_{\mathrm{rd}}}{\longrightarrow}G_{80} + G_{80}$,1.667 × 10-5 s-1,"cytoplasm, nucleoplasm",Gal80p/Gal80p dimer dissociation
4,$D_{G1} + G_{4}\overset{k_{\mathrm{f1\_4}}}{\longrightarrow}D_{G1}{:}G_{4D}$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal4p binding to gene
5,$D_{G1}{:}G_{4D}\overset{k_{\mathrm{r1\_4}}}{\longrightarrow}D_{G1} + G_{4}$,6.410 × 10-3 s-1,nucleoplasm,Gal4p dissociation from gene
6,$D_{G1}{:}G_{4D} + G_{80}\overset{k_{\mathrm{f2\_4}}}{\longrightarrow}D_{G1}{:}G_{4D}{:}G_{80D}$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal80p binding to gene
7,$D_{G1}{:}G_{4D}{:}G_{80D}\overset{k_{\mathrm{r2\_4}}}{\longrightarrow}D_{G1}{:}G_{4D} + G_{80}$,1.422 × 10-3 s-1,nucleoplasm,Gal80p dissociation from gene
8,$D_{G2} + G_{4}\overset{k_{\mathrm{f1\_5}}}{\longrightarrow}D_{G2}{:}G_{4D}$,3.583 × 107 M-1⋅s-1,nucleoplasm,Gal4p binding to gene
9,$D_{G2}{:}G_{4D}\overset{k_{\mathrm{r1\_5}}}{\longrightarrow}D_{G2} + G_{4}$,0.168 s-1,nucleoplasm,Gal4p dissociation from gene


# Initial conditions

In [22]:
initMolec = pickle.load(open("ysZeroGAE.pkl", "rb"))
# change molecular/unit volume to actual molecules
volScale = np.sum(B.convexHull(sim.siteLattice==reg.plasmaMembrane.idx))*sim.siteV/cellVol
def initMolecules(x):
    # convert molecules/unit volume to molecues
    counts = int(initMolec[x]*volScale)
    return counts

# place all genes randomly in the nucleoplasm

for b in ["DG1", "DG2", "DG3", "DG80", "DGrep"]:
    ops = [b+x for x in ["", "_G4d", "_G4d_G80d"]]
    spName = max(ops, key=lambda x:initMolec[x])
    print("{} in state {}".format(b, spName))
    sim.species(spName).placeNumberInto(reg.nucleoplasm, 1)
sp.DG4.placeNumberInto(reg.nucleoplasm, 1)
print("{} in state {}".format("DG4", "DG4"))

# place proteins/ metabolites

sp.G1.placeNumberInto(reg.cytoplasm, initMolecules("G1"))
print("G1 in cytoplasm: {}".format(initMolecules("G1")))
sp.G2.placeNumberInto(reg.plasmaMembrane, initMolecules("G2"))
print("G2 in plasma membrane: {}".format(initMolecules("G2")))
sp.G3.placeNumberInto(reg.cytoplasm, initMolecules("G3"))
print("G3 in cytoplasm: {}".format(initMolecules("G3")))
sp.G4d.placeNumberInto(reg.nucleoplasm, initMolecules("G4d"))
print("G4d in nucleoplasm: {}".format(initMolecules("G4d")))
sp.Grep.placeNumberInto(reg.cytoplasm, initMolecules("Grep"))
print("Grep in cytoplasm: {}".format(initMolecules("Grep")))

# place G80
# G80 is placed in the cytoplasm and nucleoplasm, separated by relative volume
cscl = reg.cytoplasm.volume/(reg.cytoplasm.volume+reg.nucleoplasm.volume)
totM = initMolecules("G80C") + initMolecules("G80")
totD = initMolecules("G80Cd") + initMolecules("G80d")
sp.G80.placeNumberInto(reg.cytoplasm, int(cscl*totM))
sp.G80.placeNumberInto(reg.nucleoplasm, int((1-cscl)*totM))
sp.G80d.placeNumberInto(reg.cytoplasm, int(cscl*totD))
sp.G80d.placeNumberInto(reg.nucleoplasm, int((1-cscl)*totD))
print("G80 in cytoplasm: {}, in nucleoplasm: {}".format(int(cscl*totM), int((1-cscl)*totM)))
print("G80d in cytoplasm: {}, in nucleoplasm: {}".format(int(cscl*totD), int((1-cscl)*totD)))

# place ribosome particles in ribosomes region

for x, y, z in np.argwhere(sim.siteLattice == reg.ribosomes.idx):
    sp.ribosome.placeParticle(x, y, z, 1)

print("ribosomes number:", np.sum(sim.siteLattice == reg.ribosomes.idx))

DG1 in state DG1_G4d_G80d
DG2 in state DG2_G4d_G80d
DG3 in state DG3_G4d_G80d
DG80 in state DG80_G4d_G80d
DGrep in state DGrep_G4d_G80d
DG4 in state DG4
G1 in cytoplasm: 111
G2 in plasma membrane: 1224
G3 in cytoplasm: 3092
G4d in nucleoplasm: 291
Grep in cytoplasm: 111
G80 in cytoplasm: 0, in nucleoplasm: 0
G80d in cytoplasm: 195, in nucleoplasm: 26
ribosomes number: 180000


# Diffusion Coefficients

In [23]:
with sim.construct():
    sim.transitionRate(None, None, None, sim.diffusionZero)

Index,Name,Value
0,$$D_\varnothing$$,0.000 m2⋅s-1


## DNA
Fix DNA in the location:

In [24]:
with sim.construct():
    for sps in sim.speciesList.matchRegex("D.*"):
        sps.diffusionRate(None, sim.diffusionZero)

## mRNA 

In [25]:
with sim.construct():
    sim.diffusionConst("mrna", mRNADiffusion, texRepr=r'D_{mRNA}', annotation='Generic mRNA')

    for mrna in sim.speciesList.matchRegex("R.*"):
        sim.transitionRate(mrna, reg.nucleoplasm, reg.cytoplasm, dc.mrna)
        sim.transitionRate(mrna, reg.cytoplasm, reg.nucleoplasm, sim.diffusionZero)
        sim.transitionRate(mrna, reg.nucleoplasm, reg.nucleoplasm, dc.mrna)
        sim.transitionRate(mrna, reg.cytoplasm, reg.cytoplasm, dc.mrna)
        sim.transitionRate(mrna, reg.ribosomes, reg.ribosomes, dc.mrna)
        sim.transitionRate(mrna, reg.ribosomes, reg.cytoplasm, dc.mrna)
        sim.transitionRate(mrna, reg.cytoplasm, reg.ribosomes, dc.mrna)

Index,Name,Value,Annotation
1,$$D_{mRNA}$$,5.000 × 10-14 m2⋅s-1,Generic mRNA


## Ribosome occlusion
after proteins get translated, it can only diffuse out of ribosomes, cant get back.

In [26]:
with sim.construct():
    sim.diffusionConst("prot", 1e-12, texRepr=r'D_{prot}', annotation='Generic protein')
    for sps in [sp.G1, sp.G2, sp.G3, sp.G3i, sp.G4, sp.G4d, sp.G80, sp.G80d, sp.G80d_G3i, sp.Grep]:
        sim.transitionRate(sps, reg.cytoplasm, reg.cytoplasm, dc.prot)
        sim.transitionRate(sps, reg.ribosomes, reg.cytoplasm, sim.diffusionFast)
        sim.transitionRate(sps, reg.cytoplasm, reg.ribosomes, sim.diffusionZero)

Index,Name,Value,Annotation
2,$$D_{prot}$$,1.000 × 10-12 m2⋅s-1,Generic protein
3,$$D_\infty$$,2.045 × 10-12 m2⋅s-1,


## Transcription Factors(G4, G80)

allow them to diffuse into the nucleoplasm, and diffuse in the nucleoplasm

In [27]:
with sim.construct():
    for sps in [sp.G4, sp.G4d, sp.G80, sp.G80d]:
        sim.transitionRate(sps, reg.nucleoplasm, reg.nucleoplasm, dc.prot)
        sim.transitionRate(sps, reg.nucleoplasm, reg.cytoplasm, dc.prot)
        sim.transitionRate(sps, reg.cytoplasm, reg.nucleoplasm, dc.prot)

## cytoplasmic protein

prevent them diffuse into nucleoplasm

In [28]:
with sim.construct():
    for sps in [sp.G1, sp.G2, sp.G3, sp.G3i, sp.G80d_G3i, sp.Grep]:
        sim.transitionRate(sps, reg.cytoplasm, reg.nucleoplasm, sim.diffusionZero) 

## Transporter

once diffuse into the membrane, can not get out.

In [29]:
with sim.construct():
    sim.transitionRate(sp.G2, reg.cytoplasm, reg.plasmaMembrane, dc.prot)
    sim.transitionRate(sp.G2, reg.plasmaMembrane, reg.cytoplasm, sim.diffusionZero)
    sim.diffusionConst("mem", 0.01e-12, texRepr=r'D_{mem}', annotation='Generic protein on membrane')
    # sp.G2.diffusionRate(reg.plasmaMembrane, dc.mem)
    sim.transitionRate(sp.G2, reg.plasmaMembrane, reg.plasmaMembrane, dc.mem)

Index,Name,Value,Annotation
4,$$D_{mem}$$,1.000 × 10-14 m2⋅s-1,Generic protein on membrane


## Ribosomes

not moving, stay in the corresponding cube

In [30]:
with sim.construct():
    for sps in sim.speciesList.matchRegex("ribosome.*"):
        sim.transitionRate(sps, None, None, sim.diffusionZero)

# RDME-ODE hybrid

In [31]:
import pyximport
pyximport.install(setup_args={ "include_dirs":np.get_include()})
from ode import RHS

In [32]:
class OdeRdmeHybridSolver:
    """Hybrid solver for Galactose switch
    
    Transport reactions and galactose metabolism handled by ODE, rest by RDME. The
    ODE system is coupled to the RDME by updating the protein counts in the ODE 
    each communication step. New proteins are added to the state unbound to
    galactose. If the number of proteins decreases, both bound and unbound 
    states are decreased by the same proportion and the galactose is
    added back as internal or external galactose. The RDME is coupled to the 
    ODE system through the internal galactose concentration. The rate of
    G3 -> G3i is updated with the internal galactose concentration each
    communication step."""
    def __init__(self, lmFile, initialExternalGalactose):
        super(OdeRdmeHybridSolver, self).__init__()
        self.GAE = initialExternalGalactose
        self.odeYs = None
        self.lastOdeEval = 0
        self.odeEvals = []
        if isinstance(lmFile, (RDMEFile, RDMESim)):
            self.rdme = lmFile
        else:
            self.rdme = RDME.File(lmFile)
        self.cellVol = self.rdme.reg.cytoplasm.volume + self.rdme.reg.nucleoplasm.volume + self.rdme.reg.plasmaMembrane.volume
        self.NAV = 6.022e23*self.cellVol
        self.buildOdeSys(
                [[["G1","GAI"],  ["G1GAI"],     1.442e5],
                [["G1GAI"],     ["G1", "GAI"], 30.708],
                [["G1GAI"],     ["G1"],        55.833],
                [["G2GAI"],     ["G2GAE"],     72.5],
                [["G2GAE"],     ["G2GAI"],     72.5],
                [["G2GAE"],     ["G2"], 39.875],
                [["G2"],        ["G2GAE"],     1.123e5*self.GAE],
                [["G2", "GAI"], ["G2GAI"],     1.123e5],
                [["G2GAI"],     ["G2", "GAI"], 39.875]])
        
                  
        self.g3actRidx = self.rdme.reaction(self.rdme.sp.G3, self.rdme.sp.G3i, self.rdme.rc.fi).idx
        self.g3actRc = self.rdme.rc.fi._toLM()
        
    def copyInitialConditions(self, cts):
        # initialization will count all G1 and G2 in ODE
        y = np.zeros(len(self.odeSpNames))
        y[self.odeSpIndex("GAI")] = 0
        y[self.odeSpIndex("G1")] = cts['countBySpecies'][self.rdme.sp.G1]/self.NAV 
        y[self.odeSpIndex("G1GAI")] = 0
        y[self.odeSpIndex("G2")] = cts['countBySpecies'][self.rdme.sp.G2]/self.NAV
        y[self.odeSpIndex("G2GAE")] = 0
        y[self.odeSpIndex("G2GAI")] = 0
        self.boundGal = self.rdmeGal(cts)
        return y
  
    def rdmeGal(self, cts):
        # get internal galactose counts from RDME and change to concentration in ODE
        return (cts['countBySpecies'][self.rdme.sp.G3i] + cts['countBySpecies'][self.rdme.sp.G80d_G3i])/self.NAV

    def rdme2odeConc(self, y0, cts):
        y = y0.copy()
        # update G1 in ODE
        g1ode = y0[self.odeSpIndex("G1")]
        g1gaiode = y0[self.odeSpIndex("G1GAI")]
        g1rdme = cts['countBySpecies'][self.rdme.sp.G1]/self.NAV
        #update criteria
        change = g1rdme-g1ode-g1gaiode
        
        if change > 0:
            y[self.odeSpIndex("G1")] = g1ode + change
        else:
            fracChange = g1rdme/(g1ode+g1gaiode)
            y[self.odeSpIndex("G1")] = g1ode*fracChange
            y[self.odeSpIndex("G1GAI")] = g1gaiode*fracChange
            y[self.odeSpIndex("GAI")] += g1gaiode*(1-fracChange)
            
        # update G2 in ODE
        g2ode = y0[self.odeSpIndex("G2")]
        g2gaiode = y0[self.odeSpIndex("G2GAI")]
        g2gaeode = y0[self.odeSpIndex("G2GAE")]
        g2rdme = cts['countBySpeciesRegion'][self.rdme.sp.G2][self.rdme.reg.plasmaMembrane]/self.NAV
        
        change = g2rdme-g2ode-g2gaiode-g2gaeode
        
        if change >= 0:
            y[self.odeSpIndex("G2")] = g2ode + change
        else:
            fracChange = g2rdme/(g2ode+g2gaiode+g2gaeode)
            # y[self.odeSpIndex("G2")] = g2ode*fracChange
            # y[self.odeSpIndex("G2GAI")] = g2gaiode*fracChange
            # y[self.odeSpIndex("GAI")] += g2gaiode*(1-fracChange)
            # y[self.odeSpIndex("G2GAE")] = g2gaeode*fracChange
            y[self.odeSpIndex("G2")] = int(g2ode*fracChange)
            y[self.odeSpIndex("G2GAI")] = int(g2gaiode*fracChange)
            y[self.odeSpIndex("GAI")] += int(g2gaiode*(1-fracChange))
            y[self.odeSpIndex("G2GAE")] = int(g2gaeode*fracChange)
        # update internal galactose in ODE
        g0 = self.boundGal
        g1 = self.rdmeGal(cts)
        y[self.odeSpIndex("GAI")] += g1-g0
        self.boundGal = g1
            
        return y
  
                   
    def hookSimulation(self, t, lattice):
        # this is for the hook simulation 
        cts = self.rdme.particleStatistics(particleLattice=lattice.getParticleLatticeView(),
                                           siteLattice=lattice.getSiteLatticeView())
        if self.odeYs is None:
            ys0 = self.copyInitialConditions(cts)
        else:
            ys0 = self.rdme2odeConc(self.odeYs, cts)
                   
        dt = t-self.lastOdeEval
        if dt>0:
            ys1 = self.stepOde(dt, ys0)
        else:
            ys1 = ys0
            
        self.odeEvals.append((t,ys1))
        self.odeYs = ys1
        self.lastOdeEval = t
            
        assocRt = self.g3actRc*ys1[self.odeSpIndex("GAI")]
        print(assocRt, type(assocRt))
        print(self.g3actRidx, type(self.g3actRidx))
        # print(self.rdme.reactionList[self.g3actRidx].rate.value)
        self.setReactionRate(self.g3actRidx, assocRt)
        # print(self.rdme.reactionList[self.g3actRidx].rate.value)
        self.print_ode_evals(t,assocRt,cts)
        
        return 0
    def print_ode_evals(self,t,assocRt,cts):
        print("="*80)
        print("t=",t)
        print("ODE")
        for i,n in enumerate(self.odeSpNames):
            print("  {:<16s}{:16.5g}".format(n,self.odeYs[i]))
        print("RDME")
        for n in self.rdme.speciesList:
            print("  {:<16s}{:16d}".format(n.name,cts['countBySpecies'][n]))
        print("new rate g3 activation: {:.3g}".format(assocRt))
        print("-"*80)
        return 
    def odeSpIndex(self, sp):
        return self.odeSpNames.index(sp)
    
    def buildOdeSys(self, rxns):
        odeSpNames = set()
        for react, prod, _ in rxns:
            odeSpNames |= set(prod) | set(react)
        nsp = len(odeSpNames)
        nrxn = len(rxns)
        stoch = np.zeros((nrxn, nsp), dtype=np.int32)
        dep = np.zeros((nrxn, nsp), dtype=np.int32)
        ks = np.zeros(nrxn, dtype=np.float64)
        self.odeSpNames = sorted(odeSpNames)
        for i, (react, prod, rate) in enumerate(rxns):
            ks[i] = rate
            for r in react:
                dep[i, self.odeSpIndex(r)] += 1
                stoch[i, self.odeSpIndex(r)] -= 1
            for p in prod:
                stoch[i, self.odeSpIndex(p)] += 1
        self.odeStoch = stoch
        self.odeDep = dep
        self.odeKs = ks
    
    def stepOde(self, t, ys0):
        rhs = RHS(self.odeStoch, self.odeDep, self.odeKs)
        ts = np.linspace(0,t, 10)
        ys = spint.odeint(rhs.evaluate, ys0, ts)
        return ys[-1,:]

In [33]:
Solver = makeSolver(IntMpdRdmeSolver, OdeRdmeHybridSolver)

solver = Solver(sim, externalGal)
sim.finalize()

In [34]:
try:
    traj = sim.run(solver=solver, cudaDevices=[0])
except Exception as e:
    print(f"An error occurred: {e}")
    traj = None

if traj is not None:
    odeData = dict(ts=np.array([x[0] for x in solver.odeEvals]),
                   ys=np.array([x[1] for x in solver.odeEvals]),
                   names=solver.odeSpNames)
    with open(traj.filename + "_ode.pkl", "wb") as f:
        pickle.dump(odeData, f)
else:
    print("Trajectory is not defined due to an error in simulation.")


2024-08-14 09:52:33) Info: Using 32 processor(s) and 1 CUDA device(s) per process.
2024-08-14 09:52:33) Info: Assigning 1.00 processor(s) and 1.00 CUDA device(s) per replicate.
2024-08-14 09:52:33) Info: Data output thread running.
0.0 <class 'numpy.float64'>
24 <class 'int'>
t= 0.0
ODE
  G1                    7.2268e-09
  G1GAI                          0
  G2                     7.969e-08
  G2GAE                          0
  G2GAI                          0
  GAI                            0
RDME
  DGrep                          0
  DGrep_G4d                      0
  DGrep_G4d_G80d                 1
  Rrep                           0
  Grep                         111
  DG1                            0
  DG1_G4d                        0
  DG1_G4d_G80d                   1
  R1                             0
  G1                           111
  DG2                            0
  DG2_G4d                        0
  DG2_G4d_G80d                   1
  R2                             0
  G2   